# Convolutional Neural Network

### Importing the libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
tf.__version__

In [ ]:
# Optional: mount Google Drive when using Colab.
try:
    from google.colab import drive
except ImportError:
    pass  # Local Jupyter: no Drive mount needed.
else:
    drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path
# Set CNN_DATASET_DIR, or edit this default to your existing dataset folder.
default_dir = '/content/drive/MyDrive/dataset' if Path('/content/drive/MyDrive').exists() else 'dataset'
DATASET_DIR = Path(os.environ.get('CNN_DATASET_DIR', default_dir))
for split in ['training_set', 'test_set']:
    for label in ['cats', 'dogs']:
        if not (DATASET_DIR / split / label).is_dir():
            raise FileNotFoundError(f'Missing folder: {DATASET_DIR / split / label}. See README.md.')

## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [ ]:
train_datagen = ImageDataGenerator(
    rescale = 1./ 255,
    shear_range = 0.2,
    zoom_range = 0.2,
    horizontal_flip = True)
training_set = train_datagen.flow_from_directory(
    str(DATASET_DIR / 'training_set'),
    target_size = (64, 64),
    batch_size = 32,
    class_mode = 'binary')

### Preprocessing the Test set

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_data = test_datagen.flow_from_directory(
        str(DATASET_DIR / 'test_set'),
        target_size=(64, 64),
        batch_size= 32,
        class_mode='binary')

## Part 2 - Building the CNN

### Initialising the CNN

In [ ]:
cnn = tf.keras.models.Sequential()

### Step 1 - Convolution

In [ ]:
cnn.add(tf.keras.layers.Input(shape=(64, 64, 3)))
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))

### Step 2 - Pooling

In [ ]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size = 2, strides = 2))

### Adding a second convolutional layer

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters = 32, kernel_size = 3, activation = 'relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size = 2, strides = 2))

### Step 3 - Flattening

In [ ]:
cnn.add(tf.keras.layers.Flatten())

### Step 4 - Full Connection

In [ ]:
cnn.add(tf.keras.layers.Dense(units =128, activation = 'relu'))

### Step 5 - Output Layer

In [ ]:
cnn.add(tf.keras.layers.Dense(units =1, activation = 'sigmoid'))

## Part 3 - Training the CNN

### Compiling the CNN

In [ ]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

### Training the CNN on the Training set and evaluating it on the Test set

In [ ]:
cnn.fit(x = training_set, validation_data = test_data, epochs = 25)

## Part 4 - Making a single prediction

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
# Change this path to the image you want to classify.
IMAGE_PATH = DATASET_DIR / 'test_set' / 'cats' / 'cat.4007.jpg'
test_image = image.load_img(IMAGE_PATH, target_size=(64, 64))
test_image = image.img_to_array(test_image).astype('float32') / 255.0
test_image = np.expand_dims(test_image, axis=0)
score = float(cnn.predict(test_image, verbose=0)[0][0])
index_to_class = {index: label for label, index in training_set.class_indices.items()}
prediction = index_to_class[int(score >= 0.5)]
print('Prediction:', prediction)
print('Model sigmoid score for class 1:', round(score, 4))

In [ ]:
print(prediction)